Taking the plots out of the file ev_experiments to safe the data 

In [92]:
from __future__ import annotations

import os
from typing import Callable, Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
import matplotlib
import glob
import pickle
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed


Pickle 

In [93]:
def load_all_saved_objects(base_dir="."):
    """
    Walk through base_dir, find all *_data.pkl inside data_{label} folders,
    and load the full dict from each pickle (arrays, DataFrames, whatever).

    Returns:
        data[label][dataset_name] = dict_from_pickle
    """
    data = {}

    for root, _, files in os.walk(base_dir):
        for fname in files:
            if not fname.endswith("_data.pkl"):
                continue

            path = os.path.join(root, fname)

            folder = os.path.basename(root)
            if not folder.startswith("data_"):
                continue 

            label = folder.replace("data_", "")
            dataset_name = fname.replace("_data.pkl", "")

            with open(path, "rb") as f:
                obj = pickle.load(f)

            if label not in data:
                data[label] = {}
            data[label][dataset_name] = obj

    return data

In [94]:
def get_scenario_style(label, idx):
    """
    Returns (color, linewidth, zorder) based on the label.
    - Baseline is always Black, thick, and on top.
    - Others use a muted (Dark2) palette.
    """
    # Muted palette (Dark2) color codes
    # [Green, Orange, Purple, Pink, Lime, Yellow, Brown, Grey]
    # muted_colors = plt.cm.Set2.colors
    muted_colors = ['darkred', 'darkblue', 'darkolivegreen', 'gold']
    
    if label == 'baseline':
        return 'red', 1, 10  # Color, LineWidth, Z-Order (draw on top)
    else:
        # Cycle through muted colors, skipping index if needed
        color = muted_colors[idx % len(muted_colors)]
        return color, 1, 5

def plot_combined_fanchart(group_name, scenario_labels, all_dfs, out_dir):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for i, label in enumerate(scenario_labels):
        if label not in all_dfs: continue
            
        X_list = all_dfs[label]['intervention']['baseline_X']
        if not X_list: continue

        mat = np.vstack(X_list)
        mean_curve = mat.mean(axis=0)
        # q10 = np.quantile(mat, 0.10, axis=0)
        t = np.arange(len(mean_curve))

        # Get style
        c, lw, z = get_scenario_style(label, i)
        
        # Plot Mean
        ax.plot(t, mean_curve, color=c, linewidth=lw, zorder=z, label=label)
        # Plot Band (Lower alpha for background)
        # ax.fill_between(t, q10, color=c, alpha=0.1, zorder=z-1)

    ax.set_title(f"{group_name}: Adoption Over Time (Mean & 10-90% Band)")
    ax.set_xlabel("Time")
    ax.set_ylabel("Adoption Share X(t)")
    ax.set_ylim(0, 1)
    ax.legend(loc='upper left', frameon=True)
    ax.grid(True, alpha=0.2, linestyle='--')

    safe_name = group_name.replace(" ", "_").replace(",", "")
    out_path = os.path.join(out_dir, f"fanchart_{safe_name}.png")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path

def plot_combined_spaghetti(group_name, scenario_labels, all_dfs, out_dir):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for i, label in enumerate(scenario_labels):
        if label not in all_dfs: continue
            
        X_list = all_dfs[label]['intervention']['baseline_X']
        if not X_list: continue
            
        # Get style
        c, lw, z = get_scenario_style(label, i)
        
        # Plot Traces (Very low alpha for 'hair' lines)
        for trace in X_list:
            t = np.arange(len(trace))
            ax.plot(t, trace, color=c, alpha=0.05, linewidth=0.5, zorder=z-2)

        # Plot Mean on top for legibility
        mat = np.vstack(X_list)
        mean_curve = mat.mean(axis=0)
        ax.plot(t, mean_curve, color=c, linewidth=lw, zorder=z, label=label)

    ax.set_title(f"{group_name}: Individual Traces (Spaghetti)")
    ax.set_xlabel("Time")
    ax.set_ylabel("Adoption Share X(t)")
    ax.set_ylim(0, 1)
    # Custom legend to avoid showing the faint alpha lines
    leg = ax.legend(loc='upper left', frameon=True)
    for lh in leg.legend_handles: 
        lh.set_alpha(1) # Make legend icons opaque

    ax.grid(True, alpha=0.2, linestyle='--')

    safe_name = group_name.replace(" ", "_").replace(",", "")
    out_path = os.path.join(out_dir, f"spaghetti_{safe_name}.png")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path

def plot_combined_sweeps(group_name, scenario_labels, all_dfs, out_dir):
    fig, ax = plt.subplots(figsize=(8, 5))
    
    for i, label in enumerate(scenario_labels):
        if label not in all_dfs: continue
            
        df = all_dfs[label]['ratio_sweep']['sweep_df']
        if df is not None and not df.empty:
            df = df.sort_values(by="ratio")
            
            # Get style
            c, lw, z = get_scenario_style(label, i)
            
            ax.plot(df["ratio"], df["X_mean"], color=c, linewidth=lw, 
                    zorder=z, label=label,  markersize=4)

    ax.set_title(f"{group_name}: Final Adoption Sensitivity")
    ax.set_xlabel("a_I / b (Payoff Ratio)")
    ax.set_ylabel("Final Adoption X*")
    ax.set_ylim(0, 1)
    ax.legend(loc='lower right', frameon=True)
    ax.grid(True, alpha=0.2, linestyle='--')

    safe_name = group_name.replace(" ", "_").replace(",", "")
    out_path = os.path.join(out_dir, f"sweep_{safe_name}.png")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path

In [95]:
# if __name__ == "__main__":
#     all_dfs = load_all_saved_objects()

#     # Example: access some specific DataFrames
#     # all_dfs["InitialAdoption0.3"]["intervention"]["baseline_df"]
#     # all_dfs["InitialAdoption0.3"]["phase"]["phase_df"]
#     # all_dfs["InitialAdoption0.3"]["ratio_sweep"]["sweep_df"]

#     # Quick sanity print
#     for label, datasets in all_dfs.items():
#         print(f"\n=== {label} ===")
#         for name, df_dict in datasets.items():
#             for key, df in df_dict.items():
                # print(f"{name}.{key}")

Fanchart

In [96]:
def plot_intervention_fanchart(
    baseline_X: List[np.ndarray],
    subsidy_X: List[np.ndarray],
    out_path: Optional[str] = None,
) -> str:
    """Plot fan charts for baseline and subsidy trials and save to file.

    Returns the file path to the saved image.
    """
    T = len(baseline_X[0]) if baseline_X else 0
    t = np.arange(T)

    def quantiles(X_list: List[np.ndarray]):
        mat = np.vstack(X_list)
        return {
            "mean": mat.mean(axis=0),
            "q10": np.quantile(mat, 0.10, axis=0),
            "final": mat[:, -1],
        }

    bq = quantiles(baseline_X)
    sq = quantiles(subsidy_X)

    fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)

    # Baseline fan chart
    ax = axes[0, 0]
    ax.fill_between(t, bq["q10"], color="steelblue", alpha=0.15, label="10%")
    for X in baseline_X:
        ax.plot(t, X, color="steelblue", alpha=0.10, linewidth=1)
    ax.plot(t, bq["mean"], color="steelblue", linewidth=2, label="mean")
    ax.set_title("Baseline adoption")
    ax.set_xlabel("Time")
    ax.set_ylabel("X(t)")
    ax.set_ylim(0, 1)
    ax.legend(loc="lower right")

    # Subsidy fan chart
    ax = axes[0, 1]
    ax.fill_between(t, sq["q10"], color="darkorange", alpha=0.15, label="10%")
    for X in subsidy_X:
        ax.plot(t, X, color="darkorange", alpha=0.10, linewidth=1)
    ax.plot(t, sq["mean"], color="darkorange", linewidth=2, label="mean")
    ax.set_title("Carbon Tax adoption")
    ax.set_xlabel("Time")
    ax.set_ylabel("X(t)")
    ax.set_ylim(0, 1)
    ax.legend(loc="lower right")

    # Histograms of final X(T)
    axes[1, 0].hist(bq["final"], bins=20, color="steelblue", alpha=0.8)
    axes[1, 0].set_title("Baseline final adoption X(T)")
    axes[1, 0].set_xlabel("X(T)")
    axes[1, 0].set_ylabel("Count")

    axes[1, 1].hist(sq["final"], bins=20, color="darkorange", alpha=0.8)
    axes[1, 1].set_title("Carbon Tax final adoption X(T)")
    axes[1, 1].set_xlabel("X(T)")
    axes[1, 1].set_ylabel("Count")

    # Save figure
    if out_path is None:
        out_path = os.path.join(os.getcwd(), "ev_intervention_fanchart.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path


In [97]:
def plot_final_histogram(
    baseline_X: List[np.ndarray],
    subsidy_X: List[np.ndarray],
    out_path: Optional[str] = None,
) -> str:
    """
    Plots a histogram of the final adoption values (X at time T).
    """
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)

    baseline_final = [x[-1] for x in baseline_X]
    subsidy_final = [x[-1] for x in subsidy_X]
    # Histograms of final X(T)
    axes[0].hist(baseline_final, bins=20, color="steelblue", alpha=0.8)
    axes[0].set_title("Baseline final adoption X(T)")
    axes[0].set_xlabel("X(T)")
    axes[0].set_ylabel("Count")

    axes[1].hist(baseline_final, bins=20, color="darkorange", alpha=0.8)
    axes[1].set_title("Carbon Tax final adoption X(T)")
    axes[1].set_xlabel("X(T)")
    axes[1].set_ylabel("Count")

    # Save figure
    if out_path is None:
        out_path = os.path.join(os.getcwd(), "ev_intervention_fanchart.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    
    return out_path

In [98]:
def plot_fanchart(traces_df: pd.DataFrame, out_path: Optional[str] = None) -> str:
    """Plot fan charts (quantile bands) for baseline vs subsidy using traces DF.

    traces_df columns: ['group', 'trial', 'time', 'X'] where group in {'baseline','subsidy'}.
    """
    if traces_df.empty:
        raise ValueError("traces_df is empty")

    groups = ["baseline", "subsidy"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 4.5), constrained_layout=True)

    for j, group in enumerate(groups):
        gdf = traces_df[traces_df["group"] == group]

        # Compute quantiles by time across trials
        q = gdf.groupby("time")["X"].quantile([0.10]).unstack(level=1)
        mean = gdf.groupby("time")["X"].mean()
        t = mean.index.to_numpy()

        ax = axes[0, j]
        ax.fill_between(t, q[0.10], color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.15, label="10–90%")

        # Overlay some traces for context (sample up to 100 trials)
        trial_ids = gdf["trial"].unique()
        rng = np.random.default_rng(123)
        sample = rng.choice(trial_ids, size=min(100, len(trial_ids)), replace=False)
        for tr in sample:
            tr_df = gdf[gdf["trial"] == tr]
            ax.plot(tr_df["time"], tr_df["X"], color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.1, linewidth=0.8)

        ax.plot(t, mean, color=("steelblue" if group == "baseline" else "darkorange"), linewidth=2, label="mean")
        ax.set_title(f"{group.capitalize()} adoption")
        ax.set_xlabel("Time")
        ax.set_ylabel("X(t)")
        ax.set_ylim(0, 1)
        ax.legend(loc="lower right")

        # Final X(T) histogram
        t_max = int(gdf["time"].max())
        final_vals = gdf[gdf["time"] == t_max].groupby("trial")["X"].mean().to_numpy()
        axes[1, j].hist(final_vals, bins=20, color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.8)
        axes[1, j].set_title(f"{group.capitalize()} final X(T)")
        axes[1, j].set_xlabel("X(T)")
        axes[1, j].set_ylabel("Count")

    if out_path is None:
        out_path = _default_plot_path("ev_intervention_fanchart.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path

Phase 

In [99]:
def plot_phase_plot(phase_df: pd.DataFrame, out_path: Optional[str] = None) -> str:
    """Plot heatmap from tidy DataFrame with columns ['X0','ratio','X_final']."""
    # Pivot to matrix for imshow
    pivot = phase_df.pivot(index="ratio", columns="X0", values="X_final").sort_index().sort_index(axis=1)
    ratios = pivot.index.to_numpy()
    X0s = pivot.columns.to_numpy()

    plt.figure(figsize=(7, 4))
    im = plt.imshow(
        pivot.to_numpy(),
        origin="lower",
        extent=[X0s[0], X0s[-1], ratios[0], ratios[-1]],
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
        cmap="plasma",
    )
    plt.colorbar(im, label="Final adopters X*")
    plt.xlabel("X0 (initial adoption)")
    plt.ylabel("a_I / b (initial payoff ratio)")
    plt.title("Network phase plot: X* over X0 and a_I/b")

 # Overlay threshold X = 1/ratio
    X_thresh = 1.0 / ratios
    X_thresh_clipped = np.clip(X_thresh, 0.0, 1.0)
    plt.plot(X_thresh_clipped, ratios, color="white", linestyle="--", linewidth=1.5, label="X = b / a_I (initial)")
    plt.legend(loc="upper right")

    if out_path is None:
        out_path = _default_plot_path("ev_phase_plot.png")
    plt.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close()
    return out_path

Spaghetti 

In [100]:
def plot_spaghetti(traces_df: pd.DataFrame, *, max_traces: int = 100, alpha: float = 0.15, out_path: Optional[str] = None) -> str:
    """Spaghetti plot from traces DF for baseline vs subsidy."""
    groups = ["baseline", "subsidy"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    rng = np.random.default_rng(123)

    for j, group in enumerate(groups):
        gdf = traces_df[traces_df["group"] == group]
        trial_ids = gdf["trial"].unique()
        sample = rng.choice(trial_ids, size=min(max_traces, len(trial_ids)), replace=False)
        ax = axes[j]
        for tr in sample:
            tr_df = gdf[gdf["trial"] == tr]
            ax.plot(tr_df["time"], tr_df["X"], color=("steelblue" if group == "baseline" else "darkorange"), alpha=alpha, linewidth=0.8)
        ax.set_title(f"{group.capitalize()} traces")
        ax.set_xlabel("Time")
        ax.set_ylabel("X(t)")
        ax.set_ylim(0, 1)

    if out_path is None:
        out_path = _default_plot_path("ev_spaghetti.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path



In [101]:
def plot_density(traces_df: pd.DataFrame, *, x_bins: int = 50, time_bins: Optional[int] = None, out_path: Optional[str] = None) -> str:
    """Time-evolving density plot (2D histogram) from traces DF."""
    groups = ["baseline", "subsidy"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

    for j, group in enumerate(groups):
        gdf = traces_df[traces_df["group"] == group]
        T = int(gdf["time"].max()) + 1
        if time_bins is None:
            bins_time = T
        else:
            bins_time = time_bins
        hb = axes[j].hist2d(gdf["time"].to_numpy(), gdf["X"].to_numpy(), bins=[bins_time, x_bins], range=[[0, T - 1], [0.0, 1.0]], cmap="magma")
        axes[j].set_title(f"{group.capitalize()} density: time vs X(t)")
        axes[j].set_xlabel("Time")
        axes[j].set_ylabel("X(t)")
        fig.colorbar(hb[3], ax=axes[j], label="count")

    if out_path is None:
        out_path = _default_plot_path("ev_density.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path

Success Probability

In [102]:
def plot_success_probability(traces_df, threshold=0.9, out_path=None):
    """Plots the percentage of runs that have reached a high adoption threshold over time."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for group in ["baseline", "subsidy"]:
        gdf = traces_df[traces_df["group"] == group]
        
        # Calculate fraction of runs > threshold at each time step
        # Group by time, count how many X > threshold
        success_rate = gdf.groupby("time")["X"].apply(lambda x: (x > threshold).mean())
        
        ax.plot(success_rate.index, success_rate.values, label=f"{group}", linewidth=2)
        
    ax.set_title(f"Probability of Success (Adoption > {threshold})")
    ax.set_xlabel("Time")
    ax.set_ylabel("Fraction of Successful Runs")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    if out_path: plt.savefig(out_path)
    plt.show()

Ratio Sweep 

In [103]:
def plot_ratio_sweep(sweep_df: pd.DataFrame, out_path: Optional[str] = None) -> str:
    """Plot X* vs ratio from a DataFrame with columns ['ratio','X_mean']."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sweep_df["ratio"], sweep_df["X_mean"], color="C0", lw=2)
    ax.set_xlabel("a_I / b (ratio)")
    ax.set_ylabel("Final adoption X*")
    ax.set_title("X* vs ratio")
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.25)
    if out_path is None:
        out_path = _default_plot_path("ev_ratio_sweep.png")
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close(fig)
    return out_path

In [104]:
if __name__ == "__main__":
    # 1. Load Data
    all_dfs = load_all_saved_objects()

    # RENAME ER_early -> baseline
    if 'ER_early' in all_dfs:
        all_dfs['baseline'] = all_dfs.pop('ER_early')
        
    print(f"\n--- Generating Individual Plots for {len(all_dfs)} scenarios ---")

    for label, datasets in all_dfs.items():
        print(f"Processing: {label}...")

        # Create directory (e.g., "plots_baseline", "plots_BA")
        plot_dir = f"plots/plots_{label}"
        os.makedirs(plot_dir, exist_ok=True)

        try:
            # --- UNPACK DATA (Using the dictionary structure you confirmed) ---
            
            # 1. Phase Plot Data
            phase_df = datasets['phase']['phase_df']
            
            # 2. Fanchart Data (Baseline vs Subsidy)
            baseline_X = datasets['intervention']['baseline_X']
            subsidy_X  = datasets['intervention']['subsidy_X']
            
            # 3. Spaghetti/Density Data
            traces_df = datasets['spaghetti_density']['traces_df']

            # --- PLOT GENERATION ---

            # Phase Plot
            plot_phase_plot(phase_df, out_path=f"{plot_dir}/phase_{label}.png")
            
            # Intervention Fanchart
            plot_final_histogram(baseline_X, subsidy_X, out_path=f"{plot_dir}/histogram_{label}.png")
            
            # Spaghetti Plot
            # plot_spaghetti(traces_df, max_traces=100, alpha=0.15, out_path=f"{plot_dir}/spaghetti_{label}.png")
            
            # Density Plot
            plot_success_probability(traces_df, out_path=f"{plot_dir}/success_{label}.png")

        except KeyError as e:
            print(f"  ! Skipped {label}: Missing key {e}")
        except Exception as e:
            print(f"  ! Error processing {label}: {e}")

    print("\nDone processing individual scenarios.")

    
    # 2. Define Groups
    groups_to_plot = {
        "Baseline_Comparison": ['baseline', 'ER_late'],
        "Infra_Start":   ['InitialInfrastructure0.15', 'InitialInfrastructure0.25'],
        "Beta_Sensitivity": ['HighBetaI3.0', 'LowBetaI1.0'],
        "Adoption_Start":   ['InitialAdoption0.3', 'InitialAdoption0.5'],
        "Topology_Comparison": ['baseline', 'BA', 'Grids']
    }

    # 3. Create Directory
    comp_dir = "plots/comparisons"
    os.makedirs(comp_dir, exist_ok=True)

    # 4. Generate Plots
    for group_name, scenarios in groups_to_plot.items():
        print(f"Processing group: {group_name}...")
        
        # Ensure baseline is included if available
        current_scenarios = scenarios.copy()
        if 'baseline' not in current_scenarios and 'baseline' in all_dfs:
            current_scenarios.insert(0, 'baseline')
        
        # Plot all three types
        try:
            plot_combined_fanchart(group_name, current_scenarios, all_dfs, comp_dir)
            plot_combined_spaghetti(group_name, current_scenarios, all_dfs, comp_dir)
            plot_combined_sweeps(group_name, current_scenarios, all_dfs, comp_dir)
        except Exception as e:
            print(f"  -> Error in {group_name}: {e}")

    print(f"\nDone! Plots saved to {comp_dir}")


--- Generating Individual Plots for 13 scenarios ---
Processing: BA...


C:\Users\rosam\AppData\Local\Temp\ipykernel_7788\3486832179.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Processing: BA_early...
Processing: BA_late...
Processing: ER...
Processing: ER_late...
Processing: Grids...
Processing: HighBetaI3.0...
Processing: InitialAdoption0.3...
Processing: InitialAdoption0.5...
Processing: InitialInfrastructure0.15...
Processing: InitialInfrastructure0.25...
Processing: LowBetaI1.0...
Processing: baseline...

Done processing individual scenarios.
Processing group: Baseline_Comparison...
Processing group: Infra_Start...
Processing group: Beta_Sensitivity...
Processing group: Adoption_Start...
Processing group: Topology_Comparison...

Done! Plots saved to plots/comparisons
